# EDA – Phân loại độ chín của chuối

Notebook dùng chung cho cả 3 dataset. Mỗi lần chạy chỉ cần dán đường dẫn dataset vào `DATASET_DIR` ở cell cấu hình.

Kết quả mỗi lần chạy:
- `eda_outputs/` : các file CSV + PNG chi tiết
- `<tên_dataset>_eda_report.txt` : bản tổng hợp dạng text để đọc nhanh
- `fingerprints/<tên_dataset>_fingerprints.csv` : vân tay ảnh (MD5 + hash thị giác), dùng để so trùng với các dataset khác

Để so trùng giữa 3 dataset: mount cả 3 vào cùng một session (2 cái upload thành Kaggle Dataset, 1 cái clone từ GitHub), rồi chạy notebook lần lượt 3 lần, mỗi lần đổi `DATASET_DIR`. File fingerprints tích lũy dần trong `/kaggle/working/fingerprints`, nên từ lần chạy thứ 2 trở đi phần so trùng chéo sẽ tự có dữ liệu để đối chiếu.

## 1. Import thư viện & cấu hình
Dán đường dẫn dataset vào `DATASET_DIR`. Cell này in sẵn danh sách thư mục đang có trong `/kaggle/input` để copy cho nhanh.

In [ ]:
from pathlib import Path
import hashlib, json, os, re, subprocess, warnings
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from PIL import Image, ImageFile

warnings.filterwarnings('ignore')
ImageFile.LOAD_TRUNCATED_IMAGES = False
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True,
                     'grid.alpha': .3, 'font.size': 10})

DATASET_DIR  = ''
GITHUB_URL   = ''
DATASET_NAME = ''
NEAR_DUP_THR = 6
FP_EXTRA_DIRS = []

WORK_DIR   = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
OUTPUT_DIR = WORK_DIR / 'eda_outputs'
FP_DIR     = WORK_DIR / 'fingerprints'
for d in (OUTPUT_DIR, FP_DIR):
    d.mkdir(parents=True, exist_ok=True)

EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.tif', '.tiff', '.webp'}
SPLIT_NAMES = {'train', 'training', 'test', 'testing', 'val', 'valid', 'validation'}
GENERIC_NAMES = SPLIT_NAMES | {'images', 'image', 'img', 'data', 'dataset', 'datasets',
                               'archive', 'content', 'input', 'kaggle', 'working', 'github_data'}

inp = Path('/kaggle/input')
if inp.exists():
    print('Thư mục đang mount trong /kaggle/input:')
    for p in sorted(inp.iterdir()):
        print(' ', p)
        if p.is_dir():
            for q in sorted(p.iterdir())[:8]:
                print('     ', q)
else:
    print('Không thấy /kaggle/input – đang chạy ngoài Kaggle')

## 2. (Tùy chọn) Clone dataset từ GitHub
Kaggle clone được trực tiếp, không cần tải thủ công, với điều kiện bật **Settings → Internet: On** (tài khoản phải verify số điện thoại). Repo clone vào `/kaggle/working` vì `/kaggle/input` chỉ đọc.

Chỉ cần điền `GITHUB_URL` ở cell trên, cell này sẽ clone rồi tự tìm thư mục chứa ảnh và gán vào `DATASET_DIR`. Bỏ trống thì cell này không làm gì.

In [ ]:
if GITHUB_URL:
    repo = re.sub(r'\.git$', '', GITHUB_URL.rstrip('/').split('/')[-1])
    dest = WORK_DIR / 'github_data' / repo
    if dest.exists():
        print('Đã clone trước đó:', dest)
    else:
        dest.parent.mkdir(parents=True, exist_ok=True)
        r = subprocess.run(['git', 'clone', '--depth', '1', GITHUB_URL, str(dest)],
                           capture_output=True, text=True)
        print(r.stdout, r.stderr)
        if r.returncode != 0:
            raise RuntimeError('Clone thất bại – kiểm tra Internet đã bật trong Settings chưa')

    imgs = [p for p in dest.rglob('*') if p.is_file() and p.suffix.lower() in EXT]
    print(f'{len(imgs):,} file ảnh trong repo')
    if imgs:
        DATASET_DIR = os.path.commonpath([str(p.parent) for p in imgs])
        print('DATASET_DIR =', DATASET_DIR)
    else:
        print('Repo không có ảnh trực tiếp – có thể dùng Git LFS, cần: !git lfs install && git lfs pull')

## 3. Xác định dataset & cấu trúc thư mục
Suy ra tên dataset từ đường dẫn (bỏ qua các tên chung như `train`, `images`), đếm số ảnh và đoán cấu trúc `split/label` hay chỉ `label`.

In [ ]:
DATASET_DIR = Path(str(DATASET_DIR).strip().strip('"').strip("'"))
assert str(DATASET_DIR) not in ('.', ''), 'Chưa dán đường dẫn vào DATASET_DIR'
assert DATASET_DIR.exists(), f'Không tìm thấy: {DATASET_DIR}'

if not DATASET_NAME:
    parts = [p for p in DATASET_DIR.parts if p not in ('/', '\\')]
    raw = next((p for p in reversed(parts) if p.lower() not in GENERIC_NAMES), parts[-1])
    DATASET_NAME = re.sub(r'[^0-9A-Za-z._-]+', '_', raw).strip('_')

REPORT_PATH = WORK_DIR / f'{DATASET_NAME}_eda_report.txt'
print('DATASET_DIR  =', DATASET_DIR)
print('DATASET_NAME =', DATASET_NAME)
print('REPORT_PATH  =', REPORT_PATH)

paths = sorted(p for p in DATASET_DIR.rglob('*') if p.is_file() and p.suffix.lower() in EXT)
assert paths, 'Không tìm thấy file ảnh nào trong đường dẫn này'
print(f'\n{len(paths):,} file ảnh')

def parse_split_class(p):
    parts = p.relative_to(DATASET_DIR).parts[:-1]
    if not parts:
        return 'all', 'unknown'
    if parts[0].lower() in SPLIT_NAMES:
        return parts[0], (parts[1] if len(parts) > 1 else 'unknown')
    if len(parts) > 1 and parts[1].lower() in SPLIT_NAMES:
        return parts[1], parts[0]
    return 'all', parts[0]

struct = pd.Series([len(p.relative_to(DATASET_DIR).parts) for p in paths]).value_counts().sort_index()
print('\nSố ảnh theo độ sâu thư mục:')
print(struct.to_string())
print('\nVài đường dẫn tương đối đầu tiên:')
for p in paths[:5]:
    print('  ', p.relative_to(DATASET_DIR))

## 4. Quét ảnh & thu thập metadata
Duyệt toàn bộ file: kiểm tra ảnh hỏng, đọc kích thước, mode, format, dung lượng và tính MD5.

In [ ]:
rows = []
for i, p in enumerate(paths):
    sp, cls = parse_split_class(p)
    r = dict(path=str(p), filename=p.name, split=sp, label=cls,
             ext=p.suffix.lower(), size_kb=p.stat().st_size / 1024,
             w=np.nan, h=np.nan, ch=np.nan, mode=None, fmt=None,
             valid=False, error=None, md5=None)
    try:
        with Image.open(p) as im:
            im.verify()
        with Image.open(p) as im:
            r.update(w=im.width, h=im.height, mode=im.mode,
                     fmt=im.format, ch=len(im.getbands()), valid=True)
        r['md5'] = hashlib.md5(p.read_bytes()).hexdigest()
    except Exception as e:
        r['error'] = f'{type(e).__name__}: {e}'
    rows.append(r)
    if (i + 1) % 2000 == 0:
        print(f'  {i+1}/{len(paths)}')

df = pd.DataFrame(rows)
df['ratio'] = df.w / df.h
df['mpx'] = df.w * df.h / 1e6
ok = df[df.valid].copy()
df.to_csv(OUTPUT_DIR / 'metadata.csv', index=False)

print(f'\nTổng: {len(df):,} | Hợp lệ: {len(ok):,} | Lỗi: {len(df) - len(ok):,}')
display(df.head())
if (~df.valid).any():
    display(df.loc[~df.valid, ['filename', 'error']])

## 5. Phân bố lớp, split & định dạng
Số ảnh mỗi lớp, mức mất cân bằng, cách chia train/test (nếu dataset có sẵn), định dạng file và color mode.

In [ ]:
LABELS = sorted(df.label.unique())
cmap = plt.get_cmap('tab10')
COLORS = {l: cmap(i % 10) for i, l in enumerate(LABELS)}

ct = df.groupby(['split', 'label']).size().unstack(fill_value=0)
display(ct)

totals = df.label.value_counts()
imbalance = totals.max() / totals.min()
print(f'Số lớp: {len(totals)} | Imbalance ratio: {imbalance:.2f}')
print(f'Số split: {df.split.nunique()} ({", ".join(sorted(df.split.unique()))})')

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

axes[0, 0].bar(totals.index, totals.values, color=[COLORS[c] for c in totals.index])
for i, v in enumerate(totals.values):
    axes[0, 0].text(i, v, f'{v:,}', ha='center', va='bottom', fontweight='bold')
axes[0, 0].set_title('Số lượng ảnh / lớp')
axes[0, 0].tick_params(axis='x', rotation=20)
axes[0, 0].yaxis.set_major_locator(MaxNLocator(integer=True))

if df.split.nunique() > 1:
    (ct.T / ct.T.sum()).plot.bar(ax=axes[0, 1], stacked=True)
    axes[0, 1].set_title('Tỷ lệ split trong từng lớp')
    axes[0, 1].legend(fontsize=8)
else:
    axes[0, 1].pie(totals.values, labels=totals.index, autopct='%.1f%%', startangle=90,
                   colors=[COLORS[c] for c in totals.index])
    axes[0, 1].set_title('Tỷ trọng các lớp (dataset chưa chia split)')
    axes[0, 1].grid(False)

ext_ct = df.ext.value_counts()
axes[1, 0].pie(ext_ct, labels=ext_ct.index, autopct='%.1f%%', startangle=90)
axes[1, 0].set_title('Định dạng file')
axes[1, 0].grid(False)

mode_ct = ok['mode'].value_counts()
axes[1, 1].bar(mode_ct.index, mode_ct.values, color='#8da0cb')
for i, v in enumerate(mode_ct.values):
    axes[1, 1].text(i, v, f'{v:,}', ha='center', va='bottom', fontsize=9)
axes[1, 1].set_title('Color mode')

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'class_balance.png', bbox_inches='tight')
plt.show()
ct.to_csv(OUTPUT_DIR / 'class_counts.csv')

## 6. Kích thước, tỷ lệ khung hình & dung lượng
Phân bố width, height, aspect ratio, file size theo từng lớp.

In [ ]:
stats_cols = ['w', 'h', 'ratio', 'mpx', 'size_kb']
display(ok[stats_cols].describe().T.round(2))

summary = ok.groupby('label')[stats_cols].agg(['min', 'median', 'mean', 'max'])
display(summary.round(2))
summary.to_csv(OUTPUT_DIR / 'size_summary.csv')

res_ct = ok.groupby(['w', 'h']).size().sort_values(ascending=False).head(10)
print('10 kích thước xuất hiện nhiều nhất:')
print(res_ct.to_string())

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, col, title in zip(axes.ravel(), ['w', 'h', 'ratio', 'size_kb'],
                          ['Width (px)', 'Height (px)', 'Aspect Ratio', 'File Size (KB)']):
    for lbl, grp in ok.groupby('label'):
        ax.hist(grp[col], bins=40, alpha=.55, label=lbl, color=COLORS[lbl], density=True)
    ax.set_title(title)
    ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'size_distributions.png', bbox_inches='tight')
plt.show()

## 7. Trích xuất đặc trưng pixel & hash thị giác
Mỗi ảnh resize 128×128 để tính brightness, contrast, saturation, mean RGB, tỷ lệ pixel vàng/xanh/nâu (liên quan trực tiếp đến độ chín của chuối).

Cùng lúc tính 2 hash thị giác 64-bit:
- `dhash`: so sánh độ sáng các pixel liền kề
- `ahash`: so sánh với độ sáng trung bình

Hai hash này không đổi khi ảnh bị resize hay nén lại, nên bắt được ảnh trùng mà MD5 bỏ sót.

In [ ]:
def bits_to_hex(bits):
    return f'{int.from_bytes(np.packbits(bits).tobytes(), "big"):016x}'

def img_feats(path):
    with Image.open(path) as im:
        rgb = im.convert('RGB')
        a = np.asarray(rgb.resize((128, 128)), np.float32) / 255
        g9 = np.asarray(rgb.convert('L').resize((9, 8)), np.float32)
        g8 = np.asarray(rgb.convert('L').resize((8, 8)), np.float32)

    g = a.mean(2)
    hsv = np.asarray(Image.fromarray((a * 255).astype(np.uint8)).convert('HSV'), np.float32)
    hue, sat, val = hsv[:, :, 0] * 360 / 255, hsv[:, :, 1] / 255, hsv[:, :, 2] / 255
    colored = (sat > .25) & (val > .2)

    return dict(
        brightness=g.mean(), contrast=g.std(),
        R=a[:, :, 0].mean(), G=a[:, :, 1].mean(), B=a[:, :, 2].mean(),
        saturation=(a.max(2) - a.min(2)).mean(),
        hue_mean=hue[colored].mean() if colored.any() else np.nan,
        yellow_frac=(colored & (hue >= 40) & (hue < 70)).mean(),
        green_frac=(colored & (hue >= 70) & (hue < 170)).mean(),
        brown_frac=((hue >= 10) & (hue < 40) & (sat > .2) & (val <= .45)).mean(),
        dark_frac=(g < .2).mean(), bright_frac=(g > .8).mean(),
        dhash=bits_to_hex((g9[:, 1:] > g9[:, :-1]).ravel()),
        ahash=bits_to_hex((g8 > g8.mean()).ravel()))

feats = pd.DataFrame([img_feats(p) for p in ok.path], index=ok.index)
ok = pd.concat([ok, feats], axis=1)
FEAT_COLS = [c for c in feats.columns if c not in ('dhash', 'ahash')]
ok.to_csv(OUTPUT_DIR / 'valid_with_features.csv', index=False)

pxs = ok.groupby('label')[FEAT_COLS].agg(['mean', 'std'])
display(pxs.round(4))
pxs.to_csv(OUTPUT_DIR / 'pixel_summary.csv')

## 8. Trực quan hóa đặc trưng pixel
Phân bố brightness / saturation / tỷ lệ màu giữa các lớp, boxplot để thấy mức biến thiên, mean RGB và ma trận tương quan.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 9))

for ax, col, t in zip(axes[0], ['brightness', 'saturation', 'yellow_frac'],
                      ['Brightness', 'Saturation', 'Tỷ lệ pixel vàng']):
    for lbl, grp in ok.groupby('label'):
        ax.hist(grp[col], bins=30, alpha=.55, label=lbl, color=COLORS[lbl], density=True)
    ax.set_title(t)
    ax.legend(fontsize=8)

box_data = [ok.loc[ok.label == l, 'brown_frac'] for l in LABELS]
bp = axes[1, 0].boxplot(box_data, patch_artist=True)
for patch, l in zip(bp['boxes'], LABELS):
    patch.set_facecolor(COLORS[l])
axes[1, 0].set_xticks(range(1, len(LABELS) + 1))
axes[1, 0].set_xticklabels(LABELS, rotation=20)
axes[1, 0].set_title('Tỷ lệ pixel nâu (vết chín/thối) theo lớp')

ok.groupby('label')[['R', 'G', 'B']].mean().plot.bar(
    ax=axes[1, 1], color=['#e41a1c', '#4daf4a', '#377eb8'])
axes[1, 1].set_title('Mean RGB / lớp')
axes[1, 1].set_ylim(0, 1)
axes[1, 1].tick_params(axis='x', rotation=20)

corr = ok[FEAT_COLS].corr()
im_ax = axes[1, 2].imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
axes[1, 2].set_xticks(range(len(FEAT_COLS)))
axes[1, 2].set_yticks(range(len(FEAT_COLS)))
axes[1, 2].set_xticklabels(FEAT_COLS, rotation=45, ha='right', fontsize=8)
axes[1, 2].set_yticklabels(FEAT_COLS, fontsize=8)
axes[1, 2].set_title('Feature Correlation')
axes[1, 2].grid(False)
fig.colorbar(im_ax, ax=axes[1, 2], fraction=.046)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'pixel_analysis.png', bbox_inches='tight')
plt.show()

## 9. Đặc trưng nào phân biệt các lớp tốt nhất
Với mỗi đặc trưng, so độ phân tán giữa trung bình các lớp với độ phân tán trong nội bộ lớp. Điểm càng cao thì đặc trưng càng tách được các mức độ chín.

In [ ]:
sep = []
for c in FEAT_COLS:
    gm = ok.groupby('label')[c].mean()
    within = ok.groupby('label')[c].std().mean()
    sep.append(dict(feature=c, between_std=gm.std(), within_std=within,
                    score=gm.std() / within if within else np.nan,
                    lo_class=gm.idxmin(), lo=gm.min(), hi_class=gm.idxmax(), hi=gm.max()))

sep = pd.DataFrame(sep).sort_values('score', ascending=False).reset_index(drop=True)
display(sep.round(4))
sep.to_csv(OUTPUT_DIR / 'feature_separability.csv', index=False)

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(sep.feature[::-1], sep.score[::-1], color='#66c2a5')
ax.set_title('Khả năng phân biệt lớp của từng đặc trưng (cao = tốt)')
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'feature_separability.png', bbox_inches='tight')
plt.show()

## 10. Trùng lặp trong nội bộ dataset
Ba mức kiểm tra:
1. **MD5** – trùng từng byte, tức bản copy y nguyên
2. **dhash giống hệt** – cùng nội dung ảnh nhưng đã bị resize/nén lại nên MD5 khác
3. **Hamming ≤ ngưỡng** – gần giống nhau (crop nhẹ, chỉnh sáng, cùng một quả chuột chụp liên tiếp)

Đồng thời kiểm tra ảnh trùng có rơi vào 2 split khác nhau (data leakage) hay bị gán 2 nhãn khác nhau.

In [ ]:
POP = np.array([bin(i).count('1') for i in range(256)], np.uint8)

def to_bytes(hex_series):
    arr = np.array([int(h, 16) for h in hex_series], dtype=np.uint64)
    return np.frombuffer(np.ascontiguousarray(arr.astype('>u8')).tobytes(), np.uint8).reshape(-1, 8)

def near_pairs(hex1, hex2=None, thr=NEAR_DUP_THR, chunk=512):
    b1 = to_bytes(hex1)
    same = hex2 is None
    b2 = b1 if same else to_bytes(hex2)
    out = []
    for s in range(0, len(b1), chunk):
        blk = b1[s:s + chunk]
        d = POP[np.bitwise_xor(blk[:, None, :], b2[None, :, :])].sum(2, dtype=np.int16)
        if same:
            for r in range(len(blk)):
                d[r, :s + r + 1] = 999
        i, j = np.where(d <= thr)
        if len(i):
            out.append(np.stack([i + s, j, d[i, j]], 1))
    return np.concatenate(out) if out else np.zeros((0, 3), int)

def clusters(n, pairs):
    parent = list(range(n))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for i, j, _ in pairs:
        a, b = find(int(i)), find(int(j))
        if a != b:
            parent[a] = b
    return pd.Series([find(i) for i in range(n)])

md5_groups = ok.groupby('md5').size()
dups = ok[ok.duplicated('md5', keep=False)].sort_values('md5')
n_md5_dup_groups = int((md5_groups > 1).sum())
n_unique_md5 = int(ok.md5.nunique())

span_split = ok.groupby('md5').split.nunique()
span_label = ok.groupby('md5').label.nunique()
leak_groups = int((span_split > 1).sum())
conflict_groups = int((span_label > 1).sum())

dhash_dup_groups = int((ok.groupby('dhash').size() > 1).sum())
n_unique_dhash = int(ok.dhash.nunique())

print(f'Ảnh hợp lệ: {len(ok):,}')
print(f'MD5 trùng: {n_md5_dup_groups} nhóm / {len(dups):,} file  → còn {n_unique_md5:,} ảnh unique ({n_unique_md5/len(ok)*100:.1f}%)')
print(f'dhash trùng khít: {dhash_dup_groups} nhóm → còn {n_unique_dhash:,} ảnh khác nhau về thị giác')
print(f'Nhóm MD5 nằm ở nhiều split (leakage): {leak_groups}')
print(f'Nhóm MD5 bị gán nhiều nhãn khác nhau: {conflict_groups}')

pairs = near_pairs(ok.dhash.values)
ok = ok.reset_index(drop=True)
grp = clusters(len(ok), pairs)
n_visual_unique = int(grp.nunique())
print(f'\nCặp gần trùng (Hamming ≤ {NEAR_DUP_THR}): {len(pairs):,}')
print(f'Sau khi gộp các cụm gần trùng: {n_visual_unique:,} ảnh thực sự khác nhau '
      f'({n_visual_unique/len(ok)*100:.1f}% của {len(ok):,})')

if len(dups):
    dups.to_csv(OUTPUT_DIR / 'duplicates.csv', index=False)
    display(dups[['label', 'split', 'filename', 'md5']].head(10))

if len(pairs):
    nd = pd.DataFrame({
        'dist': pairs[:, 2],
        'file_a': ok.filename.values[pairs[:, 0]], 'label_a': ok.label.values[pairs[:, 0]],
        'split_a': ok.split.values[pairs[:, 0]], 'path_a': ok.path.values[pairs[:, 0]],
        'file_b': ok.filename.values[pairs[:, 1]], 'label_b': ok.label.values[pairs[:, 1]],
        'split_b': ok.split.values[pairs[:, 1]], 'path_b': ok.path.values[pairs[:, 1]]})
    nd['same_label'] = nd.label_a == nd.label_b
    nd.sort_values('dist').to_csv(OUTPUT_DIR / 'near_duplicates.csv', index=False)
    print(f'\nCặp gần trùng nhưng KHÁC NHÃN: {int((~nd.same_label).sum()):,}')
    display(nd.sort_values('dist').head(10)[['dist', 'label_a', 'file_a', 'label_b', 'file_b']])
else:
    nd = pd.DataFrame(columns=['dist', 'label_a', 'label_b', 'same_label'])

## 11. Biểu đồ & ảnh minh họa phần trùng lặp

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

per_label = pd.DataFrame({
    'tổng': ok.groupby('label').size(),
    'unique MD5': ok.groupby('label').md5.nunique(),
    'unique dhash': ok.groupby('label').dhash.nunique()})
per_label.plot.bar(ax=axes[0], color=['#bbbbbb', '#66c2a5', '#fc8d62'])
axes[0].set_title('Tổng vs unique theo lớp')
axes[0].tick_params(axis='x', rotation=20)
axes[0].legend(fontsize=8)

if len(nd):
    axes[1].hist(nd.dist, bins=range(0, NEAR_DUP_THR + 2), color='#e78ac3', align='left', rwidth=.8)
    axes[1].set_title(f'Phân bố khoảng cách Hamming (≤ {NEAR_DUP_THR})')
    axes[1].set_xlabel('Hamming distance')
else:
    axes[1].text(.5, .5, 'Không có cặp gần trùng', ha='center', va='center')
    axes[1].set_title('Gần trùng')

sizes = [n_visual_unique, len(ok) - n_visual_unique]
axes[2].pie(sizes, labels=[f'Khác nhau\n{sizes[0]:,}', f'Trùng/gần trùng\n{sizes[1]:,}'],
            autopct='%.1f%%', startangle=90, colors=['#66c2a5', '#fc8d62'])
axes[2].set_title('Tỷ lệ ảnh thực sự khác nhau')
axes[2].grid(False)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'duplicate_analysis.png', bbox_inches='tight')
plt.show()

if len(nd):
    show = nd.sort_values('dist').head(4)
    fig, axes = plt.subplots(len(show), 2, figsize=(7, 3.4 * len(show)))
    axes = np.atleast_2d(axes)
    for r, (_, row) in enumerate(show.iterrows()):
        for c, (pth, lb, fn) in enumerate([(row.path_a, row.label_a, row.file_a),
                                           (row.path_b, row.label_b, row.file_b)]):
            with Image.open(pth) as im:
                axes[r, c].imshow(im.convert('RGB'))
            axes[r, c].set_title(f'{lb} / {fn}\nHamming = {row.dist}', fontsize=8)
            axes[r, c].axis('off')
    fig.suptitle('Các cặp ảnh gần trùng nhất trong dataset', fontweight='bold')
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / 'duplicate_samples.png', bbox_inches='tight')
    plt.show()

## 12. So trùng với các dataset khác
Lưu vân tay của dataset hiện tại, rồi đối chiếu với các file `*_fingerprints.csv` đã có từ những lần chạy trước (và các thư mục thêm trong `FP_EXTRA_DIRS` nếu bạn upload fingerprints thành một Kaggle Dataset riêng).

Lần chạy đầu tiên sẽ chưa có gì để so — chạy tiếp dataset thứ 2, thứ 3 thì phần này mới có kết quả.

In [ ]:
fp_cols = ['filename', 'split', 'label', 'md5', 'dhash', 'ahash', 'w', 'h', 'path']
fp = ok[fp_cols].copy()
fp.insert(0, 'dataset', DATASET_NAME)
fp_file = FP_DIR / f'{DATASET_NAME}_fingerprints.csv'
fp.to_csv(fp_file, index=False)
print('Đã lưu vân tay:', fp_file)

others = [f for f in sorted(FP_DIR.glob('*_fingerprints.csv')) if f.resolve() != fp_file.resolve()]
for d in FP_EXTRA_DIRS:
    others += [f for f in sorted(Path(d).rglob('*_fingerprints.csv'))
               if f.resolve() != fp_file.resolve()]

cross_rows, cross_detail = [], []
if not others:
    print('Chưa có dataset nào khác để so. Chạy lại notebook với DATASET_DIR của dataset tiếp theo.')

for f in others:
    o = pd.read_csv(f)
    name = str(o.dataset.iloc[0]) if 'dataset' in o and len(o) else f.stem.replace('_fingerprints', '')

    md5_hit = ok.merge(o, on='md5', suffixes=('', '_o'))
    dhash_hit = ok.merge(o, on='dhash', suffixes=('', '_o'))
    p = near_pairs(ok.dhash.values, o.dhash.values)

    matched_here = len(set(p[:, 0])) if len(p) else 0
    row = dict(dataset_khac=name, so_anh_ben_kia=len(o),
               trung_md5=int(md5_hit.md5.nunique()),
               trung_dhash=int(dhash_hit.dhash.nunique()),
               cap_gan_trung=len(p), anh_bi_trung=matched_here,
               ty_le_trung_pct=round(matched_here / len(ok) * 100, 2))
    cross_rows.append(row)
    print(f'\n--- {DATASET_NAME}  vs  {name} ---')
    print(f'Ảnh bên kia: {len(o):,}')
    print(f'Trùng MD5: {row["trung_md5"]} hash | Trùng dhash khít: {row["trung_dhash"]} hash')
    print(f'Cặp gần trùng (Hamming ≤ {NEAR_DUP_THR}): {len(p):,}')
    print(f'→ {matched_here:,}/{len(ok):,} ảnh ({row["ty_le_trung_pct"]}%) có ít nhất 1 ảnh giống bên {name}')

    if len(p):
        det = pd.DataFrame({
            'dataset_khac': name, 'dist': p[:, 2],
            'file_here': ok.filename.values[p[:, 0]], 'label_here': ok.label.values[p[:, 0]],
            'path_here': ok.path.values[p[:, 0]],
            'file_other': o.filename.values[p[:, 1]], 'label_other': o.label.values[p[:, 1]],
            'path_other': o.path.values[p[:, 1]]})
        cross_detail.append(det)
        print('Bảng chéo nhãn của các cặp trùng:')
        display(pd.crosstab(det.label_here, det.label_other))

cross = pd.DataFrame(cross_rows)
if len(cross):
    display(cross)
    cross.to_csv(OUTPUT_DIR / 'cross_dataset_overlap.csv', index=False)
if cross_detail:
    cd = pd.concat(cross_detail).sort_values('dist')
    cd.to_csv(OUTPUT_DIR / 'cross_dataset_pairs.csv', index=False)
    display(cd.head(10)[['dataset_khac', 'dist', 'label_here', 'file_here', 'label_other', 'file_other']])

    show = cd[cd.path_other.apply(lambda x: Path(str(x)).exists())].head(4)
    if len(show):
        fig, axes = plt.subplots(len(show), 2, figsize=(7, 3.4 * len(show)))
        axes = np.atleast_2d(axes)
        for r, (_, row) in enumerate(show.iterrows()):
            for c, (pth, tag) in enumerate([(row.path_here, f'{DATASET_NAME} / {row.label_here}'),
                                            (row.path_other, f'{row.dataset_khac} / {row.label_other}')]):
                with Image.open(pth) as im:
                    axes[r, c].imshow(im.convert('RGB'))
                axes[r, c].set_title(f'{tag}\nHamming = {row.dist}', fontsize=8)
                axes[r, c].axis('off')
        fig.suptitle('Ảnh trùng giữa các dataset', fontweight='bold')
        plt.tight_layout()
        fig.savefig(OUTPUT_DIR / 'cross_dataset_samples.png', bbox_inches='tight')
        plt.show()

## 13. Ảnh mẫu từng lớp
Xem ảnh thật để kiểm tra nhãn có hợp lý không và các mức độ chín có khác nhau rõ ràng không.

In [ ]:
rng = np.random.default_rng(231)
for lbl in LABELS:
    sub = ok[ok.label == lbl]
    if not len(sub):
        continue
    idx = rng.choice(len(sub), min(12, len(sub)), replace=False)
    samples = sub.iloc[idx]
    fig, axes = plt.subplots(3, 4, figsize=(14, 10))
    for ax, (_, row) in zip(axes.ravel(), samples.iterrows()):
        with Image.open(row.path) as im:
            ax.imshow(im.convert('RGB'))
        ax.set_title(f'{row.filename}\n{int(row.w)}×{int(row.h)}', fontsize=8)
        ax.axis('off')
    for ax in axes.ravel()[len(samples):]:
        ax.axis('off')
    fig.suptitle(f'Samples – {lbl}', fontweight='bold')
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / f'samples_{re.sub(r"[^0-9A-Za-z_-]+", "_", lbl)}.png', bbox_inches='tight')
    plt.show()

## 14. Xuất báo cáo tổng hợp
Gom toàn bộ số liệu ở trên thành một file text đặt tên theo dataset: `<tên_dataset>_eda_report.txt`.

In [ ]:
def pc(x, total):
    return f'{x/total*100:.1f}%' if total else '-'

L = []
A = L.append
A(f'Báo cáo EDA dataset – phân loại độ chín của chuối')
A(f'Dataset : {DATASET_NAME}')
A(f'Đường dẫn: {DATASET_DIR}')
A(f'Thời điểm: {datetime.now():%Y-%m-%d %H:%M}')
A('')

A('TỔNG QUAN DATASET')
A(f'- Tổng số file ảnh: {len(df):,}')
A(f'- Ảnh hợp lệ: {len(ok):,} ({pc(len(ok), len(df))})' +
  ('' if len(ok) == len(df) else f' – {len(df)-len(ok):,} file lỗi/không đọc được'))
A(f'- Số lớp: {len(LABELS)} ({", ".join(LABELS)})')
for e, n in df.ext.value_counts().items():
    A(f'- Định dạng {e}: {n:,} ({pc(n, len(df))})')
for m, n in ok['mode'].value_counts().items():
    A(f'- Color mode {m}: {n:,} ({pc(n, len(ok))})')
A('')

A('PHÂN BỐ LỚP & CÂN BẰNG')
for l, n in totals.items():
    A(f'- {l}: {n:,} ảnh ({pc(n, len(df))})')
A(f'- Tỷ lệ mất cân bằng: {imbalance:.2f}x ({totals.idxmax()} nhiều nhất, {totals.idxmin()} ít nhất)')
A('- Mức mất cân bằng ' + ('KHÔNG đáng kể' if imbalance < 1.2 else
                           'NHẸ, chấp nhận được nhưng nên dùng class_weight' if imbalance < 1.5 else
                           'ĐÁNG KỂ, nên dùng class_weight hoặc oversampling' if imbalance < 3 else
                           'NẶNG, cần cân bằng lại dữ liệu trước khi train'))
A('')

A('PHÂN BỐ SPLIT')
if df.split.nunique() > 1:
    for sp, n in df.split.value_counts().items():
        detail = ' / '.join(f'{l} {v:,}' for l, v in ct.loc[sp][ct.loc[sp] > 0].items())
        A(f'- {sp}: {n:,} ảnh ({pc(n, len(df))}) – {detail}')
    A(f'- Số nhóm MD5 xuất hiện ở nhiều split: {leak_groups}' +
      (' – có DATA LEAKAGE giữa các split' if leak_groups else ' – không có leakage theo MD5'))
else:
    A('- Dataset chưa chia sẵn train/test, toàn bộ ảnh nằm chung một thư mục theo lớp')
    A('- Cần tự chia train/val/test (stratified theo nhãn) sau khi đã loại ảnh trùng')
A('')

A('TRÙNG LẶP TRONG DATASET')
A(f'- Trùng từng byte (MD5): {n_md5_dup_groups} nhóm / {len(dups):,} file')
A(f'- Số ảnh unique theo MD5: {n_unique_md5:,}/{len(ok):,} ({pc(n_unique_md5, len(ok))})')
A(f'- Số ảnh khác nhau theo dhash: {n_unique_dhash:,} (bắt được cả ảnh đã resize/nén lại)')
A(f'- Cặp gần trùng Hamming ≤ {NEAR_DUP_THR}: {len(nd):,}')
A(f'- Sau khi gộp cụm gần trùng còn {n_visual_unique:,} ảnh thực sự khác nhau ({pc(n_visual_unique, len(ok))})')
dup_rate = 1 - n_visual_unique / len(ok)
A(f'- Tỷ lệ dư thừa tổng thể: {dup_rate*100:.1f}%' +
  (' – không đáng kể' if dup_rate < .02 else
   ' – ở mức chấp nhận được, nên lọc trước khi train' if dup_rate < .10 else
   ' – CAO, bắt buộc deduplicate trước khi chia tập'))
if len(nd):
    A(f'- Cặp gần trùng nhưng gán KHÁC NHÃN: {int((~nd.same_label).sum()):,}' +
      (' – cần soát lại nhãn ở các cặp này' if int((~nd.same_label).sum()) else ''))
A(f'- Nhóm MD5 bị gán nhiều nhãn khác nhau: {conflict_groups}')
A('')

A('TRÙNG LẶP GIỮA CÁC DATASET')
if len(cross):
    for _, r in cross.iterrows():
        A(f'- vs {r.dataset_khac} ({r.so_anh_ben_kia:,} ảnh): trùng MD5 {r.trung_md5} hash, '
          f'trùng dhash {r.trung_dhash} hash, {r.cap_gan_trung:,} cặp gần trùng')
        A(f'  → {r.anh_bi_trung:,}/{len(ok):,} ảnh ({r.ty_le_trung_pct}%) có ảnh giống bên {r.dataset_khac}')
    worst = cross.ty_le_trung_pct.max()
    A(f'- Mức trùng cao nhất với một dataset khác: {worst}%')
    A('- Đánh giá: ' + ('các dataset gần như độc lập, gộp lại được mà không sợ lặp dữ liệu' if worst < 1 else
                        'trùng ít, gộp được nhưng nên dedup theo dhash sau khi gộp' if worst < 10 else
                        'TRÙNG NHIỀU, nếu gộp sẽ có nhiều ảnh lặp – nên chọn 1 dataset làm chính hoặc dedup kỹ' if worst < 30 else
                        'TRÙNG RẤT NHIỀU, hai dataset gần như cùng một nguồn ảnh'))
else:
    A('- Chưa có dataset nào khác để đối chiếu trong lần chạy này')
    A('- Chạy lại notebook với DATASET_DIR của dataset còn lại, fingerprints sẽ tự tích lũy để so chéo')
A('')

A('KÍCH THƯỚC ẢNH')
A(f'- Width:  min {ok.w.min():.0f} – max {ok.w.max():.0f} – mean {ok.w.mean():.0f} – median {ok.w.median():.0f}')
A(f'- Height: min {ok.h.min():.0f} – max {ok.h.max():.0f} – mean {ok.h.mean():.0f} – median {ok.h.median():.0f}')
A(f'- Số kích thước khác nhau: {len(res_ct_all := ok.groupby(["w","h"]).size()):,}')
top_res = ok.groupby(['w', 'h']).size().sort_values(ascending=False)
A(f'- Kích thước phổ biến nhất: {int(top_res.index[0][0])}×{int(top_res.index[0][1])} '
  f'({top_res.iloc[0]:,} ảnh, {pc(top_res.iloc[0], len(ok))})')
A('- ' + ('Kích thước gần như ĐỒNG NHẤT, resize sẽ ít ảnh hưởng'
         if top_res.iloc[0] / len(ok) > .9 else
         'Kích thước KHÔNG đồng nhất → cần resize thống nhất (224×224 hoặc 256×256)'))
A(f'- Aspect ratio: mean {ok.ratio.mean():.2f}, phạm vi {ok.ratio.min():.2f} – {ok.ratio.max():.2f}')
A('- ' + ('Phần lớn ảnh gần vuông → resize vuông ít biến dạng'
         if abs(ok.ratio.mean() - 1) < .15 else
         'Ảnh lệch khỏi hình vuông → cân nhắc pad thay vì resize thẳng'))
A(f'- Dung lượng: mean {ok.size_kb.mean():.0f} KB, median {ok.size_kb.median():.0f} KB, '
  f'phạm vi {ok.size_kb.min():.0f} – {ok.size_kb.max():.0f} KB')
A('')

A('ĐẶC TRƯNG PIXEL & MÀU SẮC THEO LỚP')
for l in LABELS:
    g = ok[ok.label == l]
    A(f'- {l} ({len(g):,} ảnh):')
    A(f'    brightness {g.brightness.mean():.3f} ± {g.brightness.std():.3f} | '
      f'contrast {g.contrast.mean():.3f} | saturation {g.saturation.mean():.3f}')
    A(f'    RGB = {g.R.mean():.3f} / {g.G.mean():.3f} / {g.B.mean():.3f}')
    A(f'    pixel vàng {g.yellow_frac.mean()*100:.1f}% | xanh {g.green_frac.mean()*100:.1f}% | '
      f'nâu {g.brown_frac.mean()*100:.1f}%')
A('')

A('ĐẶC TRƯNG PHÂN BIỆT LỚP TỐT NHẤT')
for _, r in sep.head(5).iterrows():
    A(f'- {r.feature}: score {r.score:.2f} (thấp nhất ở {r.lo_class} = {r.lo:.3f}, '
      f'cao nhất ở {r.hi_class} = {r.hi:.3f})')
best = sep.iloc[0]
A(f'- Rõ nhất là {best.feature}; các đặc trưng màu tách được lớp thì feature thủ công còn dùng được, '
  'nếu score đều thấp thì nên đi thẳng CNN/transfer learning')
A('')

A('NHẬN XÉT CHUNG VÀ GỢI Ý')
A(f'- Quy mô sau khi loại trùng: {n_visual_unique:,} ảnh cho {len(LABELS)} lớp '
  f'(~{n_visual_unique//max(len(LABELS),1):,} ảnh/lớp)')
A('- ' + ('Số lượng khá ít, nên dùng transfer learning + augmentation mạnh'
         if n_visual_unique < 2000 else
         'Quy mô vừa, đủ cho fine-tuning CNN pre-trained'
         if n_visual_unique < 10000 else
         'Quy mô lớn, có thể train from scratch hoặc fine-tune thoải mái'))
if dup_rate >= .10:
    A('- Việc đầu tiên: deduplicate theo dhash rồi mới chia train/val/test')
if leak_groups:
    A('- Split sẵn có bị leakage → chia lại từ tập đã dedup, stratified theo nhãn')
if imbalance >= 1.5:
    A("- Mất cân bằng lớp → class_weight='balanced' hoặc oversampling lớp ít")
if 'RGBA' in set(ok['mode']):
    A('- Có ảnh RGBA → convert sang RGB trước khi train')
if ok['mode'].nunique() > 1:
    A(f'- Color mode không thống nhất ({", ".join(map(str, ok["mode"].unique()))}) → ép convert RGB trong pipeline')
if len(cross) and cross.ty_le_trung_pct.max() >= 10:
    A('- Trùng nhiều với dataset khác → nếu gộp 3 nguồn, phải dedup chéo trước khi chia tập, '
      'không thì ảnh giống nhau sẽ nằm cả ở train và test')
A('')

A('=' * 80)
A(f'  Output files: {OUTPUT_DIR}/')
for f in sorted(OUTPUT_DIR.iterdir()):
    A(f'  - {f.name}')
A(f'  Fingerprints: {fp_file}')
A('=' * 80)

report = '\n'.join(L)
REPORT_PATH.write_text(report, encoding='utf-8')
print(report)
print('\nĐã lưu:', REPORT_PATH)